# Find meaningful two and three word phrases in your corpus like "feminist theory" or "social movement" that carry more meaning than single words alone.
What Chai says

N-gramming should be done after stopword removal and lemmatization

Set a minimum frequency cutoff — Chai suggests starting at 5

Only keep n-grams that appear frequently enough to be meaningful

Bigrams and trigrams are enough — going beyond that is rarely useful

#### 1. Imports
Using this: https://www.geeksforgeeks.org/nlp/n-gram-language-modelling-with-nltk/

In [16]:
import pandas as pd
import ast
from nltk.util import ngrams
from nltk.probability import FreqDist

#### 2. Loading the Dataset

In [17]:
df_h1 = pd.read_csv("/Users/sophiehamann/master-thesis-code/data/output_files/09_h1_lemmatized.csv")
df_h1["tokens"] = df_h1["tokens"].apply(ast.literal_eval)

#### 3. Combining all tokens into one list

In [18]:
# join all token lists into one big list
all_tokens = [token for tokens in df_h1["tokens"] for token in tokens]
print(f"Total tokens: {len(all_tokens)}")

Total tokens: 52451


#### 4. Now I am generating bigrams and trigrams
Using this: https://www.geeksforgeeks.org/nlp/n-gram-language-modelling-with-nltk/

In [19]:
# bigrams = two word phrases
bigrams  = list(ngrams(all_tokens, 2))
# trigrams = three word phrases
trigrams = list(ngrams(all_tokens, 3))

print(f"Total bigrams:  {len(bigrams)}")
print(f"Total trigrams: {len(trigrams)}")

Total bigrams:  52450
Total trigrams: 52449


#### 5. Counting the Frequencies

In [20]:
bigram_freq  = FreqDist(bigrams)
trigram_freq = FreqDist(trigrams)

# look at the 20 most common ones
print("Top 20 bigrams:")
print(bigram_freq.most_common(20))
print("---")
print("Top 20 trigrams:")
print(trigram_freq.most_common(20))

Top 20 bigrams:
[(('new', 'york'), 75), (('black', 'woman'), 35), (('live', 'new'), 32), (('issue', 'heresy'), 31), (('woman', 'work'), 27), (('feminist', 'art'), 26), (('would', 'like'), 24), (('woman', 'not'), 23), (('art', 'politics'), 20), (('year', 'old'), 19), (('woman', 'movement'), 18), (('work', 'together'), 18), (('york', 'city'), 17), (('woman', 'art'), 17), (('heresy', 'collective'), 16), (('woman', 'make'), 16), (('year', 'ago'), 16), (('woman', 'music'), 16), (('art', 'world'), 14), (('woman', 'woman'), 14)]
---
Top 20 trigrams:
[(('live', 'new', 'york'), 28), (('new', 'york', 'city'), 17), (('woman', 'aware', 'historically'), 11), (('heresy', 'feminist', 'art'), 11), (('feminist', 'art', 'politics'), 11), (('queed', 'ob', 'spade'), 11), (('heresy', 'structure', 'collective'), 10), (('collective', 'ida', 'applebroog'), 10), (('patsy', 'beckert', 'joan'), 10), (('heresy', 'devote', 'examination'), 9), (('devote', 'examination', 'art'), 9), (('structure', 'collective', 'fem

#### 6. Applying a minimum frequency cutoff
Using this: https://www.geeksforgeeks.org/python/python-dictionary-comprehension/

In [21]:
# following Chai's recommendation — only keep ngrams that appear at least 5 times
min_frequency = 5

filtered_bigrams  = {gram: freq for gram, freq in bigram_freq.items() if freq >= min_frequency}
filtered_trigrams = {gram: freq for gram, freq in trigram_freq.items() if freq >= min_frequency}

print(f"Bigrams  after cutoff: {len(filtered_bigrams)}")
print(f"Trigrams after cutoff: {len(filtered_trigrams)}")

Bigrams  after cutoff: 249
Trigrams after cutoff: 52


#### 7. Saving the Results

In [22]:
# save bigrams
df_bigrams = pd.DataFrame(
    filtered_bigrams.items(),
    columns=["bigram", "frequency"]
).sort_values("frequency", ascending=False)

# save trigrams
df_trigrams = pd.DataFrame(
    filtered_trigrams.items(),
    columns=["trigram", "frequency"]
).sort_values("frequency", ascending=False)

df_bigrams.to_csv("/Users/sophiehamann/master-thesis-code/data/output_files/10_h1_bigrams.csv", index=False)
df_trigrams.to_csv("/Users/sophiehamann/master-thesis-code/data/output_files/10_h1_trigrams.csv", index=False)

print("done")
print(df_bigrams.head(10))
print(df_trigrams.head(10))

done
              bigram  frequency
57       (new, york)         75
40    (black, woman)         35
72       (live, new)         32
84   (issue, heresy)         31
59     (woman, work)         27
23   (feminist, art)         26
31     (would, like)         24
55      (woman, not)         23
4    (art, politics)         20
174      (year, old)         19
                            trigram  frequency
23                (live, new, york)         28
21                (new, york, city)         17
50               (queed, ob, spade)         11
6      (woman, aware, historically)         11
15        (feminist, art, politics)         11
14          (heresy, feminist, art)         11
12           (patsy, beckert, joan)         10
3   (heresy, structure, collective)         10
9     (collective, ida, applebroog)         10
1        (devote, examination, art)          9
